# Risk-Aware Portfolio Allocation Dashboard

## 1. Introduction
This notebook walks through an end-to-end decision-support workflow for dynamically allocating between SPY and SHY using forward-risk forecasts.

## 2. Investor Objective
The target user is a risk-aware long-term investor who wants stronger downside control than a static benchmark allocation.

In [ ]:
import numpy as np

from src.config import get_default_config
from src.data_loader import get_aligned_market_data
from src.features import build_feature_frame
from src.labels import build_label_frame
from src.model import run_walk_forward_model
from src.strategy import run_strategy_from_predictions
from src.backtest import run_backtest
from src.metrics import summarize_backtest_result
from src.sensitivity import run_threshold_sensitivity, summarize_threshold_results
from src.plots import (
    plot_equity_curves,
    plot_drawdowns,
    plot_weights,
    plot_predictions_vs_actuals,
    plot_threshold_sensitivity_sharpe,
    plot_threshold_sensitivity_drawdown,
    plot_threshold_sensitivity_turnover,
)

## 3. Data
Load and validate daily SPY/SHY data from the ingestion layer.

In [ ]:
config = get_default_config()
market_data = get_aligned_market_data(config=config, use_cache=True, save_processed=True)
market_data.prices.head()

## 4. Features
Build leakage-safe trailing predictors from historical prices and returns only.

In [ ]:
features = build_feature_frame(prices=market_data.prices, returns=market_data.returns, config=config)
features.tail()

## 5. Label
Create the forward 5-day realized volatility target for SPY.

In [ ]:
labels = build_label_frame(returns=market_data.returns, config=config)
labels.tail()

## 6. Walk-Forward Modeling
Train on historical windows and predict only on future out-of-sample windows.

In [ ]:
model_result = run_walk_forward_model(
    features=features,
    labels=labels,
    config=config,
    model_name="linear_regression",
)
model_result.predictions.head()

## 7. Strategy Rule
Map predicted risk into defensive or risk-on SPY/SHY weights.

In [ ]:
strategy_result = run_strategy_from_predictions(model_result.predictions, config=config)
strategy_result.weights.head()

## 8. Backtest
Run next-day-application backtest with transaction costs on rebalance dates.

In [ ]:
backtest_result = run_backtest(
    returns=market_data.returns,
    strategy_result=strategy_result,
    config=config,
)
summary_table = summarize_backtest_result(backtest_result)
summary_table

## 9. Benchmark Comparison
Compare dynamic strategy outcomes against static 80/20 and 60/40 benchmark paths.

In [ ]:
plot_equity_curves(
    strategy_equity=backtest_result.strategy_equity_curve,
    benchmark_equity_curves=backtest_result.benchmark_equity_curves,
)

In [ ]:
plot_drawdowns(
    strategy_equity=backtest_result.strategy_equity_curve,
    benchmark_equity_curves=backtest_result.benchmark_equity_curves,
)

In [ ]:
plot_weights(backtest_result.strategy_daily_weights)

In [ ]:
plot_predictions_vs_actuals(model_result.predictions, model_result.actuals)

## 10. Threshold Sensitivity
Use threshold grids as robustness analysis (not threshold tuning).

In [ ]:
threshold_values = np.linspace(0.10, 0.30, 9).tolist()
sensitivity_results = run_threshold_sensitivity(
    predictions=model_result.predictions,
    returns=market_data.returns,
    config=config,
    threshold_values=threshold_values,
)
sensitivity_results

In [ ]:
plot_threshold_sensitivity_sharpe(sensitivity_results)

In [ ]:
plot_threshold_sensitivity_drawdown(sensitivity_results)

In [ ]:
plot_threshold_sensitivity_turnover(sensitivity_results)

Sensitivity interpretation should focus on trade-offs (drawdown, turnover, cost, upside participation), not selecting a single "best" threshold from history.

## 11. Limitations and Future Work
- Volatility forecasting remains uncertain across regimes.
- Threshold choice changes strategy behavior and costs.
- SPY/SHY is intentionally simple and may not generalize to broader universes.
- Future work: broader assets, macro features, adaptive thresholds, classification-based regimes.